In [1]:
# Stateless
# # Initialize the client pointing to Google's OpenAI-compatible endpoint
# import os

# from openai import OpenAI


# client = OpenAI(
#     api_key=os.getenv("GEMINI_API_KEY"), 
#     base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
# )

# response = client.chat.completions.create(
#     model="gemini-3.5-flash",  # alternative: "gemini-3.5-pro"
#     messages=[
#         {"role": "system", "content": "You are a poet"},
#         {"role": "user", "content": "Sing me a sonnet"}
#     ]
# )
# print(response.choices[0].message.content)

# Stateful
import os

from google import genai

client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))
chat = client.chats.create(model="gemini-3.5-flash") 
response1= chat.send_message("You are a poet. Sing me a sonnet.") 
print(response1.text)
response2= chat.send_message("What was the 2nd line of the sonnet?")
print(response2.text)


Come, gentle soul, and hear my quiet song,
Which drifts like mist upon the evening breeze,
Where thoughts of gold and silver shadows throng,
And music whispers through the ancient trees.

I weave a song of shadow and of light,
To capture fleeting beauty as it flies,
To pierce the heavy curtain of the night,
And paint the colors of the morning skies.

Though earthly kingdoms crumble into dust,
And fleeting years like shadows steal away,
In silent ink we place our sacred trust,
To keep the warmth of summer in the grey.

So listen close, and let this music ring,
For while you hear, the heart will ever sing.
The second line of the sonnet was:

**"Which drifts like mist upon the evening breeze,"**


In [2]:
from dotenv import load_dotenv
load_dotenv()

def llm(prompt: str, model: str = "gemini-3.5-flash") -> str:
    """
    Generate a response from the LLM using the provided prompt and model.
    
    Args:
        prompt (str): The input prompt to send to the LLM.
        model (str): The model to use for generating the response. Default is "gemini-3.5-flash".
        
    Returns:
        str: The generated response from the LLM.
    """
    client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))
    chat = client.chats.create(model=model)
    response = chat.send_message(prompt)
    return response.text



In [3]:
context = """
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we're still accepting submissions.

Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

What is the video/zoom link to the stream for the "Office Hours" or live/workshop sessions?
The zoom link is only published to instructors/presenters/TAs. Students participate via YouTube Live and submit questions to Slido.

Cloud alternatives with GPU
Check the quota and reset cycle carefully. Potential options include Google Colab, Kaggle, Databricks.
"""

In [4]:
question = "I just discovered the course. Can I join now?"

prompt = f"""
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."

Question:
{question}

Context:
{context}
"""

In [5]:
answer = llm(prompt)
print(answer)

Yes, you can still join. However, if you want to receive a certificate, you need to submit your project while submissions are still being accepted. You can also just start learning and submitting homework (while the form is open) without registering.


In [6]:
#def rag(question):
    # search_results = search(question)
    # user_prompt = build_prompt(question, search_results)
    # return llm(user_prompt)

In [7]:
import requests

docs_url = "https://datatalks.club/faq/json/courses.json"
response = requests.get(docs_url)
courses_raw = response.json()

In [8]:
documents = []
url_prefix = "https://datatalks.club/faq"

for course in courses_raw:
    course_url = f"""{url_prefix}{course["path"]}"""

    course_response = requests.get(course_url)
    course_response.raise_for_status()
    course_data = course_response.json()

    documents.extend(course_data)

len(documents)

1350

In [9]:
documents[0]

{'id': '9e508f2212',
 'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: When does the course start?',
 'answer': "A new cohort runs roughly January–April every year. For the current cohort's exact start date and registration link, check the [course repo README](https://github.com/DataTalksClub/data-engineering-zoomcamp).\n\n- Register via the link in the course repo before the cohort starts.\n- Join the [course Telegram channel](https://t.me/dezoomcamp) for announcements.\n- Join DataTalks.Club's [Slack](https://datatalks.club/docs/general/slack/) and the `#course-data-engineering` channel."}

In [10]:
from minsearch import Index

index = Index(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"]
)

index.fit(documents)

In [ ]:
question = "I just discovered the course. Can I join now?"

search_results = index.search(
    question,
    boost_dict={"question": 2.0, "section": 0.5},
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)

search_results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you c

In [ ]:
# results = index.search(
#     question,
#     num_results=5,
#     boost_dict={"question": 2.0, "section": 0.5}
# )
# results
[doc["question"] for doc in search_results]


['I just discovered the course. Can I still join?',
 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
 'How should I start the course and follow the weekly workflow?',
 'When will the course be offered next?']

In [17]:
def search(question, course="llm-zoomcamp"):
    boost_dict = {"question": 2.0, "section": 0.5}
    filter_dict = {"course": course}

    return index.search(
        question,
        boost_dict=boost_dict,
        filter_dict=filter_dict,
        num_results=5
    )
search_results = search(question)
search_results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you c